In [ ]:

import os, sys, subprocess, pkgutil, warnings, time, random, re, math

NEED_REPAIR = False
try:
    import numpy as _np_test
    import pandas as _pd_test
except Exception as e:
    NEED_REPAIR = True
    print("Detected numpy/pandas compatibility problem:")
    print(e)

REPAIR_MARKER = "/content/.gpc_deps_repaired"

if NEED_REPAIR and not os.path.exists(REPAIR_MARKER):
    print("\nRepairing core scientific stack. Runtime will restart automatically...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
        "numpy==1.26.4",
        "pandas==2.2.2",
        "scipy==1.11.4",
        "scikit-learn==1.4.2",
        "matplotlib==3.8.4",
        "seaborn==0.13.2",
        "openpyxl==3.1.5"
    ])
    with open(REPAIR_MARKER, "w") as f:
        f.write("repaired")
    print("\nDependencies repaired. Restarting runtime now...")
    os.kill(os.getpid(), 9)

import sys, subprocess
import importlib.util

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name.split('<')[0].split('=')[0].split('>')[0]

    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

install_if_missing("catboost")
install_if_missing("shap")
install_if_missing("openpyxl")
install_if_missing("seaborn")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("mealpy")
install_if_missing("skillmetrics")
install_if_missing("matplotlib")
install_if_missing("pandas")


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.inspection import PartialDependenceDisplay
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
sns.set(style="whitegrid", context="notebook")
np.random.seed(42)
random.seed(42)


import mealpy
print("Mealpy version:", mealpy.__version__)

try:
    from mealpy.swarm_based.DBO import OriginalDBO
    print("Successfully imported OriginalDBO.")
except Exception as e:
    print(f"Warning: Could not import DBO ({e}). Falling back to PSO...")
    from mealpy.swarm_based.PSO import OriginalPSO as OriginalDBO

try:
    from mealpy import FloatVar
    MEALPY_NEW_API = True
except Exception:
    FloatVar = None
    MEALPY_NEW_API = False

print("Mealpy new API:", MEALPY_NEW_API)



FILE_PATH = "/content/Data.xlsx"

QUICK_MODE = True

if QUICK_MODE:
    DBO_EPOCH = 8
    DBO_POP_SIZE = 8
    CATBOOST_DEFAULT_ITER = 500
else:
    DBO_EPOCH = 40
    DBO_POP_SIZE = 25
    CATBOOST_DEFAULT_ITER = 900

N_SPLITS_OUTER = 5
N_SPLITS_INNER = 4
RANDOM_STATE = 42


if not os.path.exists(FILE_PATH):
    print("File not found at:", FILE_PATH)
    print("Upload the file first using:")
    print("from google.colab import files")
    print("uploaded = files.upload()")
    raise FileNotFoundError(f"{FILE_PATH} not found.")

df = pd.read_excel(FILE_PATH)
print("\nOriginal shape:", df.shape)
print("Original columns:")
print(df.columns.tolist())


def clean_col(col):
    col = str(col).strip()
    col = re.sub(r"\s+", " ", col)
    return col

df.columns = [clean_col(c) for c in df.columns]

column_mapping = {
    "Fly Ash amount (kg/m3)": "Fly Ash amount (kg/m3)",
    "GGBFS amount (kg/m3)": "GGBFS amount (kg/m3)",
    "NaOH molar concentration": "NaOH molar concentration",
    "NaOH amount (kg/m3)": "NaOH amount (kg/m3)",
    "Na2SiO3 amount (kg/m3)": "Na2SiO3 amount (kg/m3)",
    "Extra water": "Extra water",
    "Course Agg. (kg/m3)": "Course Agg. (kg/m3)",
    "Coarse Agg. (kg/m3)": "Course Agg. (kg/m3)",
    "Fine Agg. (kg/m3)": "Fine Agg. (kg/m3)",
    "Recycled Agg. (kg/m3)": "Recycled Agg. (kg/m3)",
    "Curing Temprature (C)": "Curing Temprature (C)",
    "Curing Temperature (C)": "Curing Temprature (C)",
    "Curing Temprature(C)": "Curing Temprature (C)",
    "Curing Time (hr)": "Curing Time (hr)",
    "age of testing (Days)": "age of testing (Days)",
    "Age of testing (Days)": "age of testing (Days)",
    "Compressive Strength (Mpa)": "Compressive Strength (Mpa)",
    "Compressive Strength (MPa)": "Compressive Strength (Mpa)",
}

df.columns = [column_mapping.get(c, c) for c in df.columns]

required_cols = [
    "Fly Ash amount (kg/m3)",
    "GGBFS amount (kg/m3)",
    "NaOH molar concentration",
    "NaOH amount (kg/m3)",
    "Na2SiO3 amount (kg/m3)",
    "Extra water",
    "Course Agg. (kg/m3)",
    "Fine Agg. (kg/m3)",
    "Recycled Agg. (kg/m3)",
    "Curing Temprature (C)",
    "Curing Time (hr)",
    "age of testing (Days)",
    "Compressive Strength (Mpa)"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required_cols].copy()


for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\nMissing values before cleaning:")
print(df.isnull().sum())

df = df.dropna().reset_index(drop=True)

before_exact = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_exact = len(df)

print(f"\nExact duplicates removed: {before_exact - after_exact}")

df_rounded = df.copy()
for c in required_cols:
    df_rounded[c] = df_rounded[c].round(3)

before_near = len(df_rounded)
df_rounded = df_rounded.drop_duplicates().reset_index(drop=True)
after_near = len(df_rounded)

df = df_rounded.copy()

print(f"Near duplicates removed after rounding: {before_near - after_near}")
print("Shape after cleaning:", df.shape)


eps = 1e-8

df["Binder total"] = df["Fly Ash amount (kg/m3)"] + df["GGBFS amount (kg/m3)"]

df["Alkaline total"] = (
    df["NaOH amount (kg/m3)"] + df["Na2SiO3 amount (kg/m3)"]
)

df["Activator/Binder"] = df["Alkaline total"] / (df["Binder total"] + eps)

df["Na2SiO3/NaOH"] = (
    df["Na2SiO3 amount (kg/m3)"] / (df["NaOH amount (kg/m3)"] + eps)
)

df["Water/Binder"] = df["Extra water"] / (df["Binder total"] + eps)

df["Agg/Binder"] = (
    df["Course Agg. (kg/m3)"] +
    df["Fine Agg. (kg/m3)"] +
    df["Recycled Agg. (kg/m3)"]
) / (df["Binder total"] + eps)

df["FA/GGBFS"] = (
    df["Fly Ash amount (kg/m3)"] /
    (df["GGBFS amount (kg/m3)"] + eps)
)


mix_cols = [
    "Fly Ash amount (kg/m3)",
    "GGBFS amount (kg/m3)",
    "NaOH molar concentration",
    "NaOH amount (kg/m3)",
    "Na2SiO3 amount (kg/m3)",
    "Extra water",
    "Course Agg. (kg/m3)",
    "Fine Agg. (kg/m3)",
    "Recycled Agg. (kg/m3)",
    "Curing Temprature (C)",
    "Curing Time (hr)"
]

mix_df = df[mix_cols].round(3).astype(str)
df["Mix_ID"] = mix_df.agg("_".join, axis=1)

print("\nTotal rows:", len(df))
print("Unique Mix_ID:", df["Mix_ID"].nunique())

target_col = "Compressive Strength (Mpa)"
feature_cols = [c for c in df.columns if c not in [target_col, "Mix_ID"]]

X = df[feature_cols].copy()
y = df[target_col].copy()
groups = df["Mix_ID"].copy()

print("\nFeatures used:")
for c in feature_cols:
    print(" -", c)


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def willmott_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_mean = np.mean(y_true)
    numerator = np.sum((y_pred - y_true) ** 2)
    denominator = np.sum(
        (np.abs(y_pred - y_mean) + np.abs(y_true - y_mean)) ** 2
    )
    if denominator == 0:
        return np.nan
    return 1 - numerator / denominator

def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE(%)": mape(y_true, y_pred),
        "WI": willmott_index(y_true, y_pred)
    }

def build_baseline_models():
    return {
        "Random Forest": RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "SVR": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(
                kernel="rbf",
                C=100,
                epsilon=0.1,
                gamma="scale"
            ))
        ]),

        "CatBoost": CatBoostRegressor(
            iterations=CATBOOST_DEFAULT_ITER,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            random_strength=1.0,
            bagging_temperature=1.0,
            loss_function="RMSE",
            eval_metric="RMSE",
            verbose=0,
            random_seed=RANDOM_STATE
        )
    }


def evaluate_catboost_group_cv(solution, X_train, y_train, groups_train, n_splits=N_SPLITS_INNER):
    depth = int(np.clip(round(solution[0]), 4, 10))
    learning_rate = float(np.clip(solution[1], 0.01, 0.30))
    iterations = int(np.clip(round(solution[2]), 200, 1200))
    l2_leaf_reg = float(np.clip(solution[3], 1.0, 15.0))
    random_strength = float(np.clip(solution[4], 0.01, 8.0))
    bagging_temperature = float(np.clip(solution[5], 0.0, 8.0))

    inner_cv = GroupKFold(n_splits=n_splits)
    rmses = []

    for tr_idx, va_idx in inner_cv.split(X_train, y_train, groups_train):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        model = CatBoostRegressor(
            depth=depth,
            learning_rate=learning_rate,
            iterations=iterations,
            l2_leaf_reg=l2_leaf_reg,
            random_strength=random_strength,
            bagging_temperature=bagging_temperature,
            loss_function="RMSE",
            eval_metric="RMSE",
            verbose=0,
            random_seed=RANDOM_STATE
        )

        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        rmse = np.sqrt(mean_squared_error(y_va, pred))
        rmses.append(rmse)

    return float(np.mean(rmses))

def solve_dbo_compatible(objective_function, lb, ub, epoch, pop_size):
    optimizer = OriginalDBO(epoch=epoch, pop_size=pop_size)

    if MEALPY_NEW_API and FloatVar is not None:
        problem = {
            "obj_func": objective_function,
            "bounds": FloatVar(lb=lb, ub=ub, name="catboost_hyperparams"),
            "minmax": "min",
        }
        best_agent = optimizer.solve(problem)

        if hasattr(best_agent, "solution"):
            best_position = np.array(best_agent.solution, dtype=float)
            if hasattr(best_agent, "target") and hasattr(best_agent.target, "fitness"):
                best_fitness = float(best_agent.target.fitness)
            else:
                best_fitness = float(objective_function(best_position))
        else:
            best_position = np.array(best_agent[0], dtype=float)
            best_fitness = float(best_agent[1])

        return best_position, best_fitness

    else:
        problem = {
            "fit_func": objective_function,
            "lb": lb,
            "ub": ub,
            "minmax": "min",
        }
        best_position, best_fitness = optimizer.solve(problem)
        return np.array(best_position, dtype=float), float(best_fitness)

def optimize_catboost_with_dbo(X_train, y_train, groups_train, epoch=DBO_EPOCH, pop_size=DBO_POP_SIZE):
    lb = [4, 0.01, 200, 1.0, 0.01, 0.0]
    ub = [10, 0.30, 1200, 15.0, 8.0, 8.0]

    def objective_function(solution):
        return evaluate_catboost_group_cv(
            solution,
            X_train,
            y_train,
            groups_train,
            n_splits=N_SPLITS_INNER
        )

    best_position, best_fitness = solve_dbo_compatible(
        objective_function=objective_function,
        lb=lb,
        ub=ub,
        epoch=epoch,
        pop_size=pop_size
    )

    best_params = {
        "depth": int(np.clip(round(best_position[0]), 4, 10)),
        "learning_rate": float(np.clip(best_position[1], 0.01, 0.30)),
        "iterations": int(np.clip(round(best_position[2]), 200, 1200)),
        "l2_leaf_reg": float(np.clip(best_position[3], 1.0, 15.0)),
        "random_strength": float(np.clip(best_position[4], 0.01, 8.0)),
        "bagging_temperature": float(np.clip(best_position[5], 0.0, 8.0)),
        "best_inner_cv_rmse": float(best_fitness)
    }

    return best_params


outer_cv = GroupKFold(n_splits=N_SPLITS_OUTER)

all_results = []
all_predictions = []
last_interpretability_pack = None
last_models = {}

model_names = ["Random Forest", "SVR", "CatBoost", "DBO-CatBoost"]

for model_name in model_names:
    print("\n" + "=" * 80)
    print(f"Evaluating model: {model_name}")
    print("=" * 80)

    fold_rows = []
    fold_predictions = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"\nFold {fold}/{N_SPLITS_OUTER}")

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()
        groups_test = groups.iloc[test_idx].copy()

        print("Train size:", X_train.shape, "| Test size:", X_test.shape)
        print("Unique train groups:", groups_train.nunique(),
              "| Unique test groups:", groups_test.nunique())

        start_time = time.time()

        if model_name != "DBO-CatBoost":
            model = build_baseline_models()[model_name]

        else:
            print("Running REAL DBO optimization for CatBoost...")
            best_params = optimize_catboost_with_dbo(
                X_train,
                y_train,
                groups_train,
                epoch=DBO_EPOCH,
                pop_size=DBO_POP_SIZE
            )
            print("Best DBO-CatBoost parameters:")
            print(best_params)

            model = CatBoostRegressor(
                depth=best_params["depth"],
                learning_rate=best_params["learning_rate"],
                iterations=best_params["iterations"],
                l2_leaf_reg=best_params["l2_leaf_reg"],
                random_strength=best_params["random_strength"],
                bagging_temperature=best_params["bagging_temperature"],
                loss_function="RMSE",
                eval_metric="RMSE",
                verbose=0,
                random_seed=RANDOM_STATE
            )

        model.fit(X_train, y_train)
        train_time = time.time() - start_time

        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        train_metrics = regression_metrics(y_train, pred_train)
        test_metrics = regression_metrics(y_test, pred_test)

        row = {
            "Model": model_name,
            "Fold": fold,
            "Train_R2": train_metrics["R2"],
            "Test_R2": test_metrics["R2"],
            "Train_RMSE": train_metrics["RMSE"],
            "Test_RMSE": test_metrics["RMSE"],
            "Train_MAE": train_metrics["MAE"],
            "Test_MAE": test_metrics["MAE"],
            "Train_MAPE(%)": train_metrics["MAPE(%)"],
            "Test_MAPE(%)": test_metrics["MAPE(%)"],
            "Train_WI": train_metrics["WI"],
            "Test_WI": test_metrics["WI"],
            "Training_Time_sec": train_time
        }

        fold_rows.append(row)

        fold_pred_df = pd.DataFrame({
            "Model": model_name,
            "Fold": fold,
            "Actual": y_test.values,
            "Predicted": pred_test,
            "Residual": y_test.values - pred_test
        })
        fold_predictions.append(fold_pred_df)

        print(
            f"Fold {fold} | Test R2={test_metrics['R2']:.4f} | "
            f"RMSE={test_metrics['RMSE']:.4f} | MAE={test_metrics['MAE']:.4f}"
        )

        if fold == N_SPLITS_OUTER:
            last_models[model_name] = model
            if model_name == "DBO-CatBoost":
                last_interpretability_pack = {
                    "model": model,
                    "X_train": X_train.copy(),
                    "X_test": X_test.copy(),
                    "y_train": y_train.copy(),
                    "y_test": y_test.copy(),
                    "pred_test": pred_test.copy()
                }

    all_results.append(pd.DataFrame(fold_rows))
    all_predictions.append(pd.concat(fold_predictions, ignore_index=True))

results_cv_df = pd.concat(all_results, ignore_index=True)
predictions_cv_df = pd.concat(all_predictions, ignore_index=True)

summary_df = results_cv_df.groupby("Model").agg({
    "Train_R2": ["mean", "std"],
    "Test_R2": ["mean", "std"],
    "Train_RMSE": ["mean", "std"],
    "Test_RMSE": ["mean", "std"],
    "Train_MAE": ["mean", "std"],
    "Test_MAE": ["mean", "std"],
    "Train_MAPE(%)": ["mean", "std"],
    "Test_MAPE(%)": ["mean", "std"],
    "Train_WI": ["mean", "std"],
    "Test_WI": ["mean", "std"],
    "Training_Time_sec": ["mean", "std"]
})

summary_df.columns = ["_".join(col).strip() for col in summary_df.columns]
summary_df = summary_df.reset_index()
summary_df = summary_df.sort_values("Test_R2_mean", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("CROSS-VALIDATED GROUPKFOLD SUMMARY")
print("=" * 80)
print(summary_df)

results_cv_df.to_csv("foldwise_results_groupkfold.csv", index=False)
summary_df.to_csv("summary_results_groupkfold.csv", index=False)
predictions_cv_df.to_csv("all_cv_predictions.csv", index=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=summary_df, x="Model", y="Test_R2_mean", palette="viridis")
plt.title("Model Comparison Based on Mean Test $R^2$ under GroupKFold")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("groupkfold_test_r2.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=summary_df, x="Model", y="Test_RMSE_mean", palette="magma")
plt.title("Model Comparison Based on Mean Test RMSE under GroupKFold")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("groupkfold_test_rmse.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=summary_df, x="Model", y="Test_MAPE(%)_mean", palette="cubehelix")
plt.title("Model Comparison Based on Mean Test MAPE under GroupKFold")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("groupkfold_test_mape.png", dpi=300)
plt.show()


plt.figure(figsize=(12, 6))
sns.boxplot(data=predictions_cv_df, x="Model", y="Residual", palette="Set2")
plt.axhline(0, color="red", linestyle="--")
plt.title("Residual Error Boxplot for All Models")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("error_boxplot_groupkfold.png", dpi=300)
plt.show()

ordered_models = summary_df["Model"].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, model_name in zip(axes, ordered_models):
    sub = predictions_cv_df[predictions_cv_df["Model"] == model_name]
    ax.scatter(sub["Actual"], sub["Predicted"], alpha=0.6, edgecolor="k")
    mn = min(sub["Actual"].min(), sub["Predicted"].min())
    mx = max(sub["Actual"].max(), sub["Predicted"].max())
    ax.plot([mn, mx], [mn, mx], "r--", lw=2)
    ax.set_title(f"{model_name}: Actual vs Predicted")
    ax.set_xlabel("Actual Strength MPa")
    ax.set_ylabel("Predicted Strength MPa")

plt.tight_layout()
plt.savefig("actual_vs_predicted_groupkfold.png", dpi=300)
plt.show()

best_model_name = summary_df.iloc[0]["Model"]
print("\nBest model based on mean Test R2:", best_model_name)

if best_model_name == "DBO-CatBoost" and last_interpretability_pack is not None:
    best_model = last_interpretability_pack["model"]
    X_train_exp = last_interpretability_pack["X_train"]
    X_test_exp = last_interpretability_pack["X_test"]
    y_train_exp = last_interpretability_pack["y_train"]
    y_test_exp = last_interpretability_pack["y_test"]
    pred_test_exp = last_interpretability_pack["pred_test"]
else:
    best_model = last_models.get(best_model_name, None)

    if best_model is None:
        print("Refitting best model on full data for interpretability only.")
        if best_model_name != "DBO-CatBoost":
            best_model = build_baseline_models()[best_model_name]
        else:
            best_params_full = optimize_catboost_with_dbo(
                X, y, groups,
                epoch=DBO_EPOCH,
                pop_size=DBO_POP_SIZE
            )
            best_model = CatBoostRegressor(
                depth=best_params_full["depth"],
                learning_rate=best_params_full["learning_rate"],
                iterations=best_params_full["iterations"],
                l2_leaf_reg=best_params_full["l2_leaf_reg"],
                random_strength=best_params_full["random_strength"],
                bagging_temperature=best_params_full["bagging_temperature"],
                loss_function="RMSE",
                eval_metric="RMSE",
                verbose=0,
                random_seed=RANDOM_STATE
            )
        best_model.fit(X, y)

    X_train_exp = X.copy()
    X_test_exp = X.copy()
    y_train_exp = y.copy()
    y_test_exp = y.copy()
    pred_test_exp = best_model.predict(X_test_exp)


if hasattr(best_model, "feature_importances_"):
    fi_df = pd.DataFrame({
        "Feature": X_train_exp.columns,
        "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=fi_df, x="Importance", y="Feature", palette="crest")
    plt.title(f"Feature Importance - {best_model_name}")
    plt.tight_layout()
    plt.savefig("feature_importance_best_model.png", dpi=300)
    plt.show()


print("\nRunning SHAP analysis...")

try:
    is_tree_model = isinstance(best_model, CatBoostRegressor) or isinstance(best_model, RandomForestRegressor)

    if is_tree_model:
        X_shap = X_test_exp.copy()
        if len(X_shap) > 300:
            X_shap = X_shap.sample(300, random_state=RANDOM_STATE)

        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_shap)

        plt.figure()
        shap.summary_plot(shap_values, X_shap, show=False)
        plt.title(f"SHAP Summary Plot - {best_model_name}")
        plt.tight_layout()
        plt.savefig("shap_summary.png", dpi=300, bbox_inches="tight")
        plt.show()

        plt.figure()
        shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
        plt.title(f"SHAP Bar Plot - {best_model_name}")
        plt.tight_layout()
        plt.savefig("shap_bar.png", dpi=300, bbox_inches="tight")
        plt.show()

    else:
        print("SHAP skipped because best model is not tree-based.")

except Exception as e:
    print("SHAP failed:", e)


print("\nRunning SHAP dependence interaction-style plots...")

try:
    is_tree_model = isinstance(best_model, CatBoostRegressor) or isinstance(best_model, RandomForestRegressor)

    if is_tree_model:
        X_dep = X_test_exp.copy()
        if len(X_dep) > 300:
            X_dep = X_dep.sample(300, random_state=RANDOM_STATE)

        explainer = shap.TreeExplainer(best_model)
        shap_values_dep = explainer.shap_values(X_dep)

        interaction_pairs = [
            ("Na2SiO3/NaOH", "NaOH molar concentration"),
            ("Activator/Binder", "GGBFS amount (kg/m3)"),
            ("Water/Binder", "age of testing (Days)"),
            ("NaOH amount (kg/m3)", "Na2SiO3 amount (kg/m3)")
        ]

        for feat_main, feat_inter in interaction_pairs:
            if feat_main in X_dep.columns and feat_inter in X_dep.columns:
                plt.figure()
                shap.dependence_plot(
                    feat_main,
                    shap_values_dep,
                    X_dep,
                    interaction_index=feat_inter,
                    show=False
                )
                plt.title(f"SHAP Dependence: {feat_main} interacting with {feat_inter}")
                safe_name = (
                    f"shap_dependence_{feat_main}_{feat_inter}"
                    .replace("/", "_")
                    .replace(" ", "_")
                    .replace("(", "")
                    .replace(")", "")
                )
                plt.tight_layout()
                plt.savefig(safe_name + ".png", dpi=300, bbox_inches="tight")
                plt.show()

        try:
            print("Trying full SHAP interaction values. This may be slow...")
            shap_interaction_values = explainer.shap_interaction_values(X_dep)

            plt.figure()
            shap.summary_plot(shap_interaction_values, X_dep, show=False)
            plt.title(f"SHAP Interaction Summary - {best_model_name}")
            plt.tight_layout()
            plt.savefig("shap_interaction_summary.png", dpi=300, bbox_inches="tight")
            plt.show()

        except Exception as e2:
            print("Full SHAP interaction matrix skipped:", e2)

except Exception as e:
    print("SHAP dependence analysis failed:", e)


candidate_pdp_features = [
    "NaOH molar concentration",
    "NaOH amount (kg/m3)",
    "Na2SiO3 amount (kg/m3)",
    "GGBFS amount (kg/m3)",
    "Curing Temprature (C)",
    "age of testing (Days)",
    "Activator/Binder",
    "Na2SiO3/NaOH",
    "Water/Binder"
]

pdp_features = [f for f in candidate_pdp_features if f in X_train_exp.columns][:6]
print("\nPDP features:", pdp_features)

try:
    fig, ax = plt.subplots(figsize=(16, 12))
    PartialDependenceDisplay.from_estimator(
        best_model,
        X_train_exp,
        features=pdp_features,
        kind="average",
        grid_resolution=40,
        ax=ax
    )
    plt.suptitle(f"Partial Dependence Plots - {best_model_name}", y=1.02)
    plt.tight_layout()
    plt.savefig("pdp_best_model.png", dpi=300, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("PDP failed:", e)


residuals = y_test_exp.values - pred_test_exp

plt.figure(figsize=(8, 6))
plt.scatter(pred_test_exp, residuals, alpha=0.7, edgecolor="k")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Strength MPa")
plt.ylabel("Residuals")
plt.title(f"Residual Plot - {best_model_name}")
plt.tight_layout()
plt.savefig("residual_plot_best_model.png", dpi=300)
plt.show()


plt.figure(figsize=(13, 10))
corr = df.drop(columns=["Mix_ID"]).corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300)
plt.show()


def centered_rmsd(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    yt = y_true - np.mean(y_true)
    yp = y_pred - np.mean(y_pred)
    return np.sqrt(np.mean((yp - yt) ** 2))

def plot_taylor_diagram(predictions_df, model_order, filename="taylor_diagram_manual.png"):
    ref_model = model_order[0]
    ref_sub = predictions_df[predictions_df["Model"] == ref_model]
    obs = ref_sub["Actual"].values
    obs_std = np.std(obs, ddof=1)

    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, polar=True)

    max_std = obs_std

    stats = []

    for model_name in model_order:
        sub = predictions_df[predictions_df["Model"] == model_name]
        actual = sub["Actual"].values
        pred = sub["Predicted"].values

        min_len = min(len(actual), len(pred))
        actual = actual[:min_len]
        pred = pred[:min_len]

        std_pred = np.std(pred, ddof=1)
        corr = np.corrcoef(actual, pred)[0, 1]
        corr = np.clip(corr, -1, 1)
        crmsd_val = centered_rmsd(actual, pred)

        stats.append({
            "Model": model_name,
            "std": std_pred,
            "corr": corr,
            "crmsd": crmsd_val
        })

        max_std = max(max_std, std_pred)

    max_std = max_std * 1.25

    corr_ticks = [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
    theta_ticks = np.arccos(corr_ticks)
    ax.set_thetagrids(np.degrees(theta_ticks), labels=[str(c) for c in corr_ticks])
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

    ax.set_rlim(0, max_std)
    ax.set_xlabel("Standard Deviation")
    ax.set_title("Taylor Diagram", y=1.08)

    ax.plot(0, obs_std, "ro", markersize=10, label="Observed")

    colors = plt.cm.tab10(np.linspace(0, 1, len(stats)))

    for color, st in zip(colors, stats):
        theta = np.arccos(st["corr"])
        radius = st["std"]
        ax.plot(theta, radius, "o", color=color, markersize=8, label=st["Model"])

    rs, ts = np.meshgrid(
        np.linspace(0, max_std, 120),
        np.linspace(0, np.pi / 2, 120)
    )
 
    crmsd_grid = np.sqrt(
        rs**2 + obs_std**2 - 2 * rs * obs_std * np.cos(ts)
    )

    contours = ax.contour(
        ts,
        rs,
        crmsd_grid,
        levels=5,
        colors="gray",
        linestyles="dotted",
        linewidths=0.8
    )
    ax.clabel(contours, inline=True, fontsize=8, fmt="%.1f")

    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(stats)

try:
    taylor_stats_df = plot_taylor_diagram(
        predictions_cv_df,
        ordered_models,
        filename="taylor_diagram_manual.png"
    )
    taylor_stats_df.to_csv("taylor_statistics.csv", index=False)
    print("\nTaylor statistics:")
    print(taylor_stats_df)
except Exception as e:
    print("Taylor diagram failed:", e)


best_pred_df = pd.DataFrame({
    "Actual": y_test_exp.values,
    "Predicted": pred_test_exp,
    "Residual": y_test_exp.values - pred_test_exp
})
best_pred_df.to_csv("best_model_predictions.csv", index=False)


leakage_audit = {
    "Total rows after cleaning": len(df),
    "Unique Mix_ID": df["Mix_ID"].nunique(),
    "Average rows per Mix_ID": len(df) / df["Mix_ID"].nunique(),
    "Number of features": len(feature_cols),
    "Outer CV": f"GroupKFold with {N_SPLITS_OUTER} folds",
    "Inner CV for DBO": f"GroupKFold with {N_SPLITS_INNER} folds",
    "DBO epoch": DBO_EPOCH,
    "DBO population size": DBO_POP_SIZE
}

leakage_audit_df = pd.DataFrame(
    list(leakage_audit.items()),
    columns=["Item", "Value"]
)
leakage_audit_df.to_csv("leakage_audit_summary.csv", index=False)

print("\n" + "=" * 80)
print("LEAKAGE AUDIT SUMMARY")
print("=" * 80)
print(leakage_audit_df)


print("\n" + "=" * 80)
print("FINAL MODEL SUMMARY")
print("=" * 80)
print(summary_df)

print("\nSaved files:")
saved_files = [
    "foldwise_results_groupkfold.csv",
    "summary_results_groupkfold.csv",
    "all_cv_predictions.csv",
    "groupkfold_test_r2.png",
    "groupkfold_test_rmse.png",
    "groupkfold_test_mape.png",
    "error_boxplot_groupkfold.png",
    "actual_vs_predicted_groupkfold.png",
    "feature_importance_best_model.png",
    "shap_summary.png",
    "shap_bar.png",
    "shap_interaction_summary.png",
    "pdp_best_model.png",
    "residual_plot_best_model.png",
    "correlation_heatmap.png",
    "taylor_diagram_manual.png",
    "taylor_statistics.csv",
    "best_model_predictions.csv",
    "leakage_audit_summary.csv"
]

for f in saved_files:
    if os.path.exists(f):
        print(" -", f)

print("\nPipeline completed successfully.")
